# 03a | SELECT, WHERE, ORDER BY

SQL is the language for pulling data out of a database. It shows up in almost every data science interview and online assessment (OA), and it is easier to score on than algorithm questions: the question types are fixed, so practice is all it takes.

The good news: everything you did in pandas has a SQL equivalent, and the data is the same four tables (`users`, `orders`, `events`, `employees`).

**After this page you can:** pull the columns and rows you want from a table and sort them. About 40 minutes.

## The code cells in this chapter are SQL

Write SQL directly in the code cells below and click Run to see the result table. Exercises compare your result with the correct one row by row.

> The practice environment uses SQLite. Interviews usually use MySQL or PostgreSQL. Everything in the first five pages is written the same way in all three; only the date functions differ, and 03f gives a comparison table.

## 1. The most basic query

```sql
SELECT col1, col2
FROM table_name;
```

`*` means all columns. `LIMIT 5` returns only the first 5 rows (like `head(5)`). SQL keywords are case-insensitive and conventionally written in uppercase; line breaks and indentation are up to you.

In [ ]:
SELECT *
FROM users
LIMIT 5;

In [ ]:
SELECT order_id, amount, amount * 7 AS amount_cny
FROM orders
LIMIT 5;

`AS` gives a column an alias. You can compute directly in `SELECT`, which is like adding a new column in pandas.

## 2. WHERE: filtering rows

| Condition | Syntax |
|---|---|
| Equal / not equal | `=` / `<>` (or `!=`) |
| And / or / not | `AND` / `OR` / `NOT` |
| In a set of values | `country IN ('US', 'CN')` |
| In a range (**both ends inclusive**) | `amount BETWEEN 20 AND 50` |
| Pattern match | `name LIKE 'A%'` (`%` matches any number of characters) |

⚠️ Two differences from Python: **equality is a single `=`**, and **strings use single quotes**.

In [ ]:
SELECT order_id, user_id, amount, status
FROM orders
WHERE amount >= 50
  AND status = 'completed';

### ✏️ Exercise 1 (fill in the blanks)

Select users whose `country` is `'US'`, with only the `name` and `channel` columns.

In [ ]:
SELECT ____, ____
FROM users
WHERE ____ = ____;

<details>
<summary><b>Answer</b></summary>

```sql
SELECT name, channel
FROM users
WHERE country = 'US';
```

</details>

### ✏️ Exercise 2 (modify the query)

The query below returns orders with an amount of at least 50. Change it to return orders with an amount **below 30** and status `completed` (all columns).

In [ ]:
SELECT *
FROM orders
WHERE amount >= 50;

<details>
<summary><b>Answer</b></summary>

```sql
SELECT *
FROM orders
WHERE amount < 30
  AND status = 'completed';
```

</details>

## 3. NULL: guaranteed to come up in interviews

`NULL` means "unknown". User Gus's `country` is NULL.

**Comparing anything with NULL never yields "true"**, including `= NULL` and `<> NULL`. So:

- To test for missing values you must write `IS NULL` / `IS NOT NULL`;
- `WHERE country <> 'US'` does **not** return rows where country is NULL.

Run the two queries below and compare the row counts:

In [ ]:
SELECT user_id, name, country
FROM users
WHERE country = NULL;      -- no rows at all: = NULL is never true

In [ ]:
SELECT user_id, name, country
FROM users
WHERE country IS NULL;

### ✏️ Exercise 3 (write it yourself)

Select users whose **country is not US**, **including users with a missing country**. Output the `user_id` and `name` columns.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

`country <> 'US'` misses Gus (his country is NULL). Add `OR country IS NULL`. The result should have 6 rows.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT user_id, name
FROM users
WHERE country <> 'US'
   OR country IS NULL;
```

</details>

## 4. ORDER BY and LIMIT

- `ORDER BY amount DESC`: sort by amount, descending (without `DESC` it is ascending);
- Separate multiple sort keys with commas: `ORDER BY dept, salary DESC`;
- `ORDER BY` followed by `LIMIT 3` gives the "top three".

**Good habit:** when taking the top N and ties are possible, add a secondary sort key so the result is deterministic.

In [ ]:
SELECT name, dept, salary
FROM employees
ORDER BY dept, salary DESC;

### ✏️ Exercise 4 (write it yourself)

From `employees`, select employees in `Eng` or `Sales` with a salary above 80. Output `name`, `dept`, `salary`, sorted by salary from high to low, with ties broken alphabetically by `name`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

`WHERE dept IN ('Eng', 'Sales') AND salary > 80`, then `ORDER BY salary DESC, name`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT name, dept, salary
FROM employees
WHERE dept IN ('Eng', 'Sales')
  AND salary > 80
ORDER BY salary DESC, name;
```

</details>

### ✏️ Exercise 5 (write it yourself)

Find the 3 `completed` orders with the highest amounts, outputting `order_id` and `amount`. Break ties by smaller `order_id` first.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Answer</b></summary>

```sql
SELECT order_id, amount
FROM orders
WHERE status = 'completed'
ORDER BY amount DESC, order_id
LIMIT 3;
```

</details>

## 5. DISTINCT

`SELECT DISTINCT col` removes duplicates, like pandas' `drop_duplicates()` / `unique()`.

In [ ]:
SELECT DISTINCT channel
FROM users;

### ✏️ Exercise 6 (write it yourself)

List orders from January 15 to January 31 (both inclusive), outputting `order_id` and `order_date`, sorted by date ascending.

Dates written as strings like `'2026-01-15'` can be compared directly.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Answer</b></summary>

```sql
SELECT order_id, order_date
FROM orders
WHERE order_date BETWEEN '2026-01-15' AND '2026-01-31'
ORDER BY order_date;
```

</details>

## Clause order and cheat sheet

The clauses of a query must be **written in a fixed order**:

```sql
SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT ...
```

| Task | pandas | SQL |
|---|---|---|
| Select columns | `df[["a", "b"]]` | `SELECT a, b` |
| Filter rows | `df[df["x"] > 1]` | `WHERE x > 1` |
| New column | `df["z"] = df["x"] * 2` | `SELECT x * 2 AS z` |
| Sort | `sort_values("x", ascending=False)` | `ORDER BY x DESC` |
| First N rows | `.head(N)` | `LIMIT N` |
| Deduplicate | `drop_duplicates()` | `SELECT DISTINCT` |
| Missing | `.isna()` | `IS NULL` |

On to **03b**.